## Rates table

In [123]:
# Step 1: Filter and Clean Invoice Data
import pandas as pd


# 🔧 Configure which sites to process
#selected_sites = ["DIT", "SPN", "SPCP","SPW","SPT","SPHU","SPTM","PVF","SPJ","CCS","SPB","SPL","SPLV","CCSG","SPCB","SPWV","FSU","SPK","SPLA","SPD","SPTG","KFC"]  # Example: update these as needed
selected_sites =['SPW','SPT','SPJ','SPN']

#selected_sites =['SPJ']

focus_columns = [
    "invoice_id", "site",'invoice_commodity_quantity', "invoice_commodity_group", "invoice_commodity_description",
    "location", "model", "unit", "rate_unit", "freight_class", "applied_rate",
    "shipment_type", "realistic_optimal_method", "xgs_rate", "historical_rate"
]

# Load the invoice input data
invoice_path = "invoice_input_data_all.xlsx"  # Update path if needed
invoice_df = pd.read_excel(invoice_path)
print("Input file from model has",invoice_df['invoice_id'].nunique())


# get invoces that have freight greater than zero
invoice_df = invoice_df[invoice_df["rate_ratio_normal_outlier"]!= 'MISSING'] # Key update here to remove rate_ratio_normal_outlier

print("Output analysis files has",invoice_df['invoice_id'].nunique())

# Only modelled for Georgia source of Georgia mill rates
invoice_georgia_df = invoice_df[invoice_df['model'] == True]
print("Output analysis file for Georga Mills has",invoice_georgia_df['invoice_id'].nunique())


# Get all modelled invoices for histotrical estimate
invoice_all_df = invoice_df


invoice_georgia_df = invoice_georgia_df[focus_columns]
invoice_all_df = invoice_all_df[focus_columns]


invoice_georgia_df["invoice_commodity_description"] = invoice_georgia_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

invoice_all_df["invoice_commodity_description"] = invoice_all_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

# Filter input invoices to selected sites
invoice_georgia_df = invoice_georgia_df[invoice_georgia_df["site"].isin(selected_sites)]
invoice_all_df = invoice_all_df[invoice_all_df["site"].isin(selected_sites)]



Input file from model has 17846
Output analysis files has 13813
Output analysis file for Georga Mills has 10300


C:\Users\nzhuw\AppData\Local\Temp/ipykernel_9628/1135092565.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invoice_all_df["invoice_commodity_description"] = invoice_all_df["invoice_commodity_description"].apply(


In [124]:
# We are calculating 3 core variables here:
# invoice_df_mills # this is for georgia_mill_rates and georgia_xgs_rates
# invoice_df_hist #this is for historical_market_rates mills and distributors

Historical market rates have to two variables historical_all_rates and historical_georgia_rates

XGS rates will have the historical_xgs_rates and 2024_xgs_rates

We will recommended a market rate recommended_market_rate


In [125]:
from numpy import average

def summarize_invoice_data(df: pd.DataFrame, group_cols: list[str], column_prefix: str = "") -> pd.DataFrame:
    # Filter required fields
    filtered = df[
        df["freight_class"].notna() &
        df["historical_rate"].notna() &
        df["xgs_rate"].notna() &
        df["invoice_commodity_quantity"].notna()
    ][[
        "site", 
        "rate_unit", 
        "invoice_commodity_group", 
        # "invoice_commodity_description",
        "freight_class", 
        "historical_rate",
        "xgs_rate",
        "invoice_commodity_quantity"
    ]].copy()

    grouped = filtered.groupby(group_cols)

    # Quantiles
    summary_avg = grouped.agg(
        historical_xgs_median = ("xgs_rate", "median"),
        historical_xgs_q3 = ("xgs_rate", lambda x: x.quantile(0.75)),
        
        historical_market_median = ("historical_rate", "median"),
        historical_market_q3 = ("historical_rate", lambda x: x.quantile(0.75)),
    )

    # Rename with prefix
    summary_avg = summary_avg.rename(columns={
        "historical_xgs_q3": f"{column_prefix}historical_xgs_q3",
        "historical_xgs_median": f"{column_prefix}historical_xgs_median",
        "historical_market_q3": f"{column_prefix}historical_market_q3",
        "historical_market_median": f"{column_prefix}historical_market_median"
    })


    # Weighted averages
    def compute_wavg(grp):
        return pd.Series({
            f"{column_prefix}historical_market_wavg": average(grp["historical_rate"], weights=grp["invoice_commodity_quantity"]),
            f"{column_prefix}historical_xgs_wavg": average(grp["xgs_rate"], weights=grp["invoice_commodity_quantity"])
        })

    summary_wavg = grouped.apply(compute_wavg)

    # Combine
    summary = pd.concat([summary_avg, summary_wavg], axis=1)
    return summary


In [126]:
summary_georgia = summarize_invoice_data(invoice_georgia_df, [
    "site", "rate_unit", "invoice_commodity_group", "freight_class"
],    column_prefix="georgia_"
)
summary_georgia


georgia_historical_xgs_median  \
site rate_unit invoice_commodity_group freight_class                                  
SPJ  CWT       1VNL                    10M                                 0.084880   
                                       1M                                  0.161562   
                                       20M                                 0.084880   
                                       2M                                  0.129970   
                                       30M                                 0.064204   
...                                                                             ...   
SPW  SQYD      1CPT                    2M                                  0.805809   
                                       3M                                  0.519059   
                                       5C                                  0.833136   
                                       5M                                  0.407181   
                                       L5C                                 0.853737   

                                                      georgia_historical_xgs_q3  \
site rate_unit invoice_commodity_group freight_class                              
SPJ  CWT       1VNL                    10M                             0.084880   
                                       1M                              0.161564   
                                       20M                             0.084880   
                                       2M                              0.129971   
                                       30M                             0.064204   
...                                                                         ...   
SPW  SQYD      1CPT                    2M                              0.805809   
                                       3M                              0.627762   
                                       5C                              0.833140   
                                       5M                              0.420097   
                                       L5C                             1.924126   

                                                      georgia_historical_market_median  \
site rate_unit invoice_commodity_group freight_class                                     
SPJ  CWT       1VNL                    10M                                    0.075766   
                                       1M                                     0.126517   
                                       20M                                    0.066257   
                                       2M                                     0.102326   
                                       30M                                    0.047456   
...                                                                                ...   
SPW  SQYD      1CPT                    2M                                     0.993678   
                                       3M                                     0.539626   
                                       5C                                     1.003434   
                                       5M                                     0.618576   
                                       L5C                                    1.403939   

                                                      georgia_historical_market_q3  \
site rate_unit invoice_commodity_group freight_class                                 
SPJ  CWT       1VNL                    10M                                0.077768   
                                       1M                                 0.151995   
                                       20M                                0.073372   
                                       2M                                 0.146110   
                                       30M                                0.047456   
...                                                                  

In [127]:
summary_georgia2 = summarize_invoice_data(invoice_georgia_df, [
    "site","invoice_commodity_group",
],    column_prefix="georgia_"
)
summary_georgia2 = summary_georgia2.reset_index()
summary_georgia2.head(2)

,site,invoice_commodity_group,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg
0,SPJ,1CBL,0.700063,1.231899,1.110548,2.142453,1.031672,0.532134
1,SPJ,1CPT,0.824475,1.139937,1.365688,2.412391,1.149326,0.762656


In [128]:
summary_sample = summarize_invoice_data(invoice_all_df, [
    "site", "rate_unit", "invoice_commodity_group", "freight_class"
],    column_prefix="all_states_"
)
summary_sample

all_states_historical_xgs_median  \
site rate_unit invoice_commodity_group freight_class                                     
SPJ  CWT       1VNL                    10M                                    0.084880   
                                       1M                                     0.161562   
                                       20M                                    0.084880   
                                       2M                                     0.129970   
                                       30M                                    0.064204   
...                                                                                ...   
SPW  SQYD      1CPT                    2M                                     0.805809   
                                       3M                                     0.519059   
                                       5C                                     0.833136   
                                       5M                                     0.407181   
                                       L5C                                    0.853737   

                                                      all_states_historical_xgs_q3  \
site rate_unit invoice_commodity_group freight_class                                 
SPJ  CWT       1VNL                    10M                                0.084880   
                                       1M                                 0.161563   
                                       20M                                0.084880   
                                       2M                                 0.129971   
                                       30M                                0.064204   
...                                                                            ...   
SPW  SQYD      1CPT                    2M                                 0.805809   
                                       3M                                 0.627762   
                                       5C                                 0.833139   
                                       5M                                 0.420097   
                                       L5C                                2.021333   

                                                      all_states_historical_market_median  \
site rate_unit invoice_commodity_group freight_class                                        
SPJ  CWT       1VNL                    10M                                       0.076877   
                                       1M                                        0.112048   
                                       20M                                       0.071310   
                                       2M                                        0.101199   
                                       30M                                       0.047456   
...                                                                                   ...   
SPW  SQYD      1CPT                    2M                                        0.993678   
                                       3M                                        0.539626   
                                       5C                                        1.006964   
                                       5M                                        0.618576   
                                       L5C                                       1.414933   

                                                      all_states_historical_market_q3  \
site rate_unit invoice_commodity_group freight_class                                    
SPJ  CWT       1VNL                    10M                                   0.077768   
                                       1M                                    0.147036   
                                       20M                                   0.079559   
                                       2M                                    0.144827   
                    

In [129]:
summary_sample.columns

Index(['all_states_historical_xgs_median', 'all_states_historical_xgs_q3',
       'all_states_historical_market_median',
       'all_states_historical_market_q3', 'all_states_historical_market_wavg',
       'all_states_historical_xgs_wavg'],
      dtype='object')

In [130]:
summary_sample2 = summarize_invoice_data(invoice_all_df, [
    "site","invoice_commodity_group",
],    column_prefix="all_states_"
)
summary_sample2 = summary_sample2.reset_index()

summary_sample2

,site,invoice_commodity_group,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,SPJ,1CBL,0.485000,0.698278,0.723750,1.336381,0.698256,0.497776
1,SPJ,1CPT,0.824478,1.151423,1.370540,2.680562,1.149648,0.770460
2,SPJ,1VNL,0.208449,0.573336,0.155884,0.602859,0.097703,0.106064
3,SPN,1CBL,0.355700,0.537150,0.854848,1.886769,0.799596,0.222275
4,SPN,1CPT,0.604758,1.363484,0.789551,2.213420,0.611734,0.433701
5,SPN,1VNL,0.185951,0.465604,0.163261,0.354539,0.075383,0.055487
6,SPT,1CBL,0.770815,1.568141,1.017568,1.345125,1.213528,0.664932
7,SPT,1CPT,0.870251,1.718213,1.390615,2.176484,1.317385,0.875873
8,SPT,1VNL,0.231243,0.453375,0.160061,0.427918,0.134574,0.130463
9,SPW,1CBL,0.534366,1.174034,1.056583,1.636838,0.635219,0.386252


In [131]:
summary_combined = pd.merge(
    summary_sample2,
    summary_georgia2,
    on=["site", "invoice_commodity_group"],
    how="outer"  # or "inner" depending on whether you want all or matching keys
)

desired_column_order = [
    "site",
    "invoice_commodity_group",

    "all_states_historical_market_median",
    "all_states_historical_market_wavg",
    "all_states_historical_market_q3",

    # "all_states_historical_xgs_median",
    # "all_states_historical_xgs_wavg",
    # "all_states_historical_xgs_q3",

    "georgia_historical_market_median",
    "georgia_historical_market_wavg",
    "georgia_historical_market_q3",

    "georgia_historical_xgs_median",
    "georgia_historical_xgs_wavg",
    "georgia_historical_xgs_q3",
]


summary_combined = summary_combined[desired_column_order]

summary_combined.columns

Index(['site', 'invoice_commodity_group',
       'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3'],
      dtype='object')

In [132]:
import pandas as pd

with pd.ExcelWriter("output/summary_output_ppt.xlsx", engine="xlsxwriter") as writer:
    summary_sample2.to_excel(writer, sheet_name="All Sites Summary", index=False)
    summary_georgia2.to_excel(writer, sheet_name="Georgia Mills Summary", index=False)
    summary_combined.to_excel(writer, sheet_name="Combined Summary", index=False)


In [133]:
summary_combined

,site,invoice_commodity_group,all_states_historical_market_median,all_states_historical_market_wavg,all_states_historical_market_q3,georgia_historical_market_median,georgia_historical_market_wavg,georgia_historical_market_q3,georgia_historical_xgs_median,georgia_historical_xgs_wavg,georgia_historical_xgs_q3
0,SPJ,1CBL,0.723750,0.698256,1.336381,1.110548,1.031672,2.142453,0.700063,0.532134,1.231899
1,SPJ,1CPT,1.370540,1.149648,2.680562,1.365688,1.149326,2.412391,0.824475,0.762656,1.139937
2,SPJ,1VNL,0.155884,0.097703,0.602859,0.144476,0.087166,0.298258,0.161562,0.100289,0.319805
3,SPN,1CBL,0.854848,0.799596,1.886769,0.770000,0.301488,0.878353,0.355700,0.180090,0.491900
4,SPN,1CPT,0.789551,0.611734,2.213420,0.780019,0.510294,1.779642,0.604757,0.421856,1.537428
5,SPN,1VNL,0.163261,0.075383,0.354539,0.149805,0.050975,0.314605,0.129432,0.047113,0.410487
6,SPT,1CBL,1.017568,1.213528,1.345125,1.007742,1.044585,1.118137,0.808533,0.710055,1.555106
7,SPT,1CPT,1.390615,1.317385,2.176484,1.383988,1.304390,1.823827,0.870253,0.902372,1.710838
8,SPT,1VNL,0.160061,0.134574,0.427918,0.153237,0.111543,0.227035,0.205300,0.126457,0.253647
9,SPW,1CBL,1.056583,0.635219,1.636838,1.053050,0.580463,1.387875,0.527304,0.381095,1.212800


In [134]:
# Step 1: Filter for numerical columns only
numerical_cols = summary_combined.select_dtypes(include='number').columns

# Step 2: Create a mask for 1VNL rows
vnl_mask = summary_combined["invoice_commodity_group"] == "1VNL"

# Step 3: Multiply numerical columns in those rows by 1.2
summary_combined.loc[vnl_mask, numerical_cols] = summary_combined.loc[vnl_mask, numerical_cols] * 1.2

print("✅ Multiplied all numeric columns for 1VNL rows by 1.2.")


✅ Multiplied all numeric columns for 1VNL rows by 1.2.


In [135]:
summary_combined

,site,invoice_commodity_group,all_states_historical_market_median,all_states_historical_market_wavg,all_states_historical_market_q3,georgia_historical_market_median,georgia_historical_market_wavg,georgia_historical_market_q3,georgia_historical_xgs_median,georgia_historical_xgs_wavg,georgia_historical_xgs_q3
0,SPJ,1CBL,0.723750,0.698256,1.336381,1.110548,1.031672,2.142453,0.700063,0.532134,1.231899
1,SPJ,1CPT,1.370540,1.149648,2.680562,1.365688,1.149326,2.412391,0.824475,0.762656,1.139937
2,SPJ,1VNL,0.187061,0.117244,0.723431,0.173371,0.104599,0.357910,0.193875,0.120347,0.383767
3,SPN,1CBL,0.854848,0.799596,1.886769,0.770000,0.301488,0.878353,0.355700,0.180090,0.491900
4,SPN,1CPT,0.789551,0.611734,2.213420,0.780019,0.510294,1.779642,0.604757,0.421856,1.537428
5,SPN,1VNL,0.195913,0.090459,0.425447,0.179766,0.061170,0.377526,0.155318,0.056536,0.492584
6,SPT,1CBL,1.017568,1.213528,1.345125,1.007742,1.044585,1.118137,0.808533,0.710055,1.555106
7,SPT,1CPT,1.390615,1.317385,2.176484,1.383988,1.304390,1.823827,0.870253,0.902372,1.710838
8,SPT,1VNL,0.192073,0.161489,0.513502,0.183884,0.133852,0.272442,0.246360,0.151749,0.304376
9,SPW,1CBL,1.056583,0.635219,1.636838,1.053050,0.580463,1.387875,0.527304,0.381095,1.212800


## Create block

In [136]:
summary = pd.concat([summary_georgia, summary_sample], ignore_index=False).reset_index()
summary 


,site,rate_unit,invoice_commodity_group,freight_class,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,SPJ,CWT,1VNL,10M,0.084880,0.084880,0.075766,0.077768,0.057670,0.084880,NaN,NaN,NaN,NaN,NaN,NaN
1,SPJ,CWT,1VNL,1M,0.161562,0.161564,0.126517,0.151995,0.152824,0.161562,NaN,NaN,NaN,NaN,NaN,NaN
2,SPJ,CWT,1VNL,20M,0.084880,0.084880,0.066257,0.073372,0.063094,0.084751,NaN,NaN,NaN,NaN,NaN,NaN
3,SPJ,CWT,1VNL,2M,0.129970,0.129971,0.102326,0.146110,0.121498,0.129970,NaN,NaN,NaN,NaN,NaN,NaN
4,SPJ,CWT,1VNL,30M,0.064204,0.064204,0.047456,0.047456,0.047456,0.064204,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,SPW,SQYD,1CPT,2M,NaN,NaN,NaN,NaN,NaN,NaN,0.805809,0.805809,0.993678,1.051058,0.902682,0.790090
151,SPW,SQYD,1CPT,3M,NaN,NaN,NaN,NaN,NaN,NaN,0.519059,0.627762,0.539626,0.953693,0.678873,0.528425
152,SPW,SQYD,1CPT,5C,NaN,NaN,NaN,NaN,NaN,NaN,0.833136,0.833139,1.006964,1.319077,1.076278,0.833136
153,SPW,SQYD,1CPT,5M,NaN,NaN,NaN,NaN,NaN,NaN,0.407181,0.420097,0.618576,0.859560,0.656591,0.405916


In [137]:
# Step 4: Create block tables for each metric (no column prefixes)

index_cols = ["site", "rate_unit", "invoice_commodity_group",]
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

def safe_pivot(metric_col, source_name):
    pivoted = summary.pivot(index=index_cols, columns="freight_class", values=metric_col).reset_index()
    
    # Ensure all freight class columns are present
    for fc in freight_classes:
        if fc not in pivoted.columns:
            pivoted[fc] = None

    # Add required template columns
    pivoted.rename(columns={
        "rate_unit": "unit",
        "invoice_commodity_group": "commodity_group",
   
    }, inplace=True)
    pivoted["site_description"] = pivoted['site']
    pivoted["unitclass"] = pivoted["unit"].apply(lambda x: "Weight" if x == "CWT" else "Area")
    pivoted["source"] = source_name

    # Reorder
    ordered_cols = ["site_description", "site", "unit", "unitclass", "commodity_group", ] + freight_classes + ["source"]
    return pivoted[ordered_cols]



In [138]:
summary.columns

Index(['site', 'rate_unit', 'invoice_commodity_group', 'freight_class',
       'georgia_historical_xgs_median', 'georgia_historical_xgs_q3',
       'georgia_historical_market_median', 'georgia_historical_market_q3',
       'georgia_historical_market_wavg', 'georgia_historical_xgs_wavg',
       'all_states_historical_xgs_median', 'all_states_historical_xgs_q3',
       'all_states_historical_market_median',
       'all_states_historical_market_q3', 'all_states_historical_market_wavg',
       'all_states_historical_xgs_wavg'],
      dtype='object')

In [139]:
def safe_pivot(metric_col, source_name):
    pivoted = summary.pivot_table(
        index=index_cols,
        columns="freight_class",
        values=metric_col,
        aggfunc="first"  # or "mean" if multiple values should be averaged
    ).reset_index()

    # Ensure all freight class columns are present
    for fc in freight_classes:
        if fc not in pivoted.columns:
            pivoted[fc] = None

    # Add required template columns
    pivoted.rename(columns={
        "rate_unit": "unit",
        "invoice_commodity_group": "commodity_group",
    }, inplace=True)
    pivoted["site_description"] = pivoted["site"]
    pivoted["unitclass"] = pivoted["unit"].apply(lambda x: "Weight" if x == "CWT" else "Area")
    pivoted["source"] = source_name

    # Reorder
    ordered_cols = ["site_description", "site", "unit", "unitclass", "commodity_group", ] + freight_classes + ["source"]
    return pivoted[ordered_cols]


In [140]:


# Generate four blocks

# Historical Georgia Mills

# Pivot each metric using safe_pivot(metric_column_name, source_name)
# Ordered: median → wavg → q3 per group

# Georgia — historical market
georgia_market_median = safe_pivot("georgia_historical_market_median", "georgia_historical_market_median")
georgia_market_wavg = safe_pivot("georgia_historical_market_wavg", "georgia_historical_market_wavg")
georgia_market_q3 = safe_pivot("georgia_historical_market_q3", "georgia_historical_market_q3")

# Georgia — xgs
georgia_xgs_median = safe_pivot("georgia_historical_xgs_median", "georgia_historical_xgs_median")
georgia_xgs_wavg = safe_pivot("georgia_historical_xgs_wavg", "georgia_historical_xgs_wavg")
georgia_xgs_q3 = safe_pivot("georgia_historical_xgs_q3", "georgia_historical_xgs_q3")

# All states — historical market
all_market_median = safe_pivot("all_states_historical_market_median", "all_states_historical_market_median")
all_market_wavg = safe_pivot("all_states_historical_market_wavg", "all_states_historical_market_wavg")
all_market_q3 = safe_pivot("all_states_historical_market_q3", "all_states_historical_market_q3")

# All states — xgs
# all_xgs_median = safe_pivot("all_states_historical_xgs_median", "all_states_historical_xgs_median")
# all_xgs_wavg = safe_pivot("all_states_historical_xgs_wavg", "all_states_historical_xgs_wavg")
# all_xgs_q3 = safe_pivot("all_states_historical_xgs_q3", "all_states_historical_xgs_q3")

# Combine all in desired order
combined_output = pd.concat([
    georgia_market_median,
    georgia_market_wavg,
    georgia_market_q3,

    georgia_xgs_median,
    georgia_xgs_wavg,
    georgia_xgs_q3,

    all_market_median,
    all_market_wavg,
    all_market_q3,

    # all_xgs_median,
    # all_xgs_wavg,
    # all_xgs_q3,
], ignore_index=True)


# Sort for visual clarity
combined_output = combined_output.sort_values(by=["commodity_group", "site", "unit", "source"]).reset_index(drop=True)

# Preview
print("✅ Combined pivot output (clean format):")
combined_output.head()


✅ Combined pivot output (clean format):


freight_class,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
2,SPJ,SPJ,SQYD,Area,1CBL,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median
4,SPJ,SPJ,SQYD,Area,1CBL,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3


In [141]:
# Step 3: Ensure All Required Columns in Combined Output

freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Ensure all freight class columns exist in the output
for col in freight_classes:
    if col not in combined_output.columns:
        combined_output[col] = None

# Ensure proper column order
ordered_cols = [
    "site_description", "site", "unit", "unitclass", "commodity_group",
] + freight_classes + ["source"]

combined_output = combined_output[ordered_cols]

# Preview the cleaned, structured result
print("✅ Final Structured Invoice Summary (Step 3):")
combined_output.head()


✅ Final Structured Invoice Summary (Step 3):


freight_class,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
2,SPJ,SPJ,SQYD,Area,1CBL,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median
4,SPJ,SPJ,SQYD,Area,1CBL,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3


## Get XGS dicounted rates an cimbine

In [142]:
# Step 4: Add Source Column and Append to Vendor Data

# Load vendor data
vendor_path = "freight_rates_operating_multi_reporting_all.csv"  # Update path if needed
vendor_df = pd.read_csv(vendor_path)

# Filter vendor freight rates to selected sites
vendor_df = vendor_df[vendor_df["site"].isin(selected_sites)]


# Add source tag
vendor_df["source"] = "vendor"

# Ensure all required freight class columns exist in vendor_df
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
for col in freight_classes:
    if col not in vendor_df.columns:
        vendor_df[col] = None

# Ensure consistent column ordering
final_cols = [
    "site_description", "site", "unit", "unitclass", "commodity_group",
] + freight_classes + ["source"]

vendor_df = vendor_df[final_cols]


In [143]:
# Constants for rate adjustment
XGS_RATE_DISCOUNT = 0.06
XGS_FUEL_SURCHARGE = 0.3
XGS_LTL_REBATE = 0.1
STARNET_REBATE = 0.025

# List of freight class columns
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Adjustment function
def adjust_rate(rate):
    inflation_rate = rate / (1 + XGS_RATE_DISCOUNT)
    fsc_rate = inflation_rate * (1 + XGS_FUEL_SURCHARGE)
    xgs_rebate = inflation_rate * XGS_LTL_REBATE
    star_net_rebate = (inflation_rate - xgs_rebate) * STARNET_REBATE
    final_rate = fsc_rate - xgs_rebate - star_net_rebate
    return final_rate

# Apply adjustments to each freight class column
for col in freight_classes:
    if col in vendor_df.columns:
        vendor_df[col] = adjust_rate(pd.to_numeric(vendor_df[col], errors='coerce'))

print("✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.")


✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.


In [144]:
# Step 6: Normalize Vendor Rates from $/CWT to $/LBS

# Identify rows where unit is CWT (used for 1VNL)
vendor_cwt_mask = (vendor_df["commodity_group"] == "1VNL") & (vendor_df["unit"] == "CWT")

# List of freight class columns to scale
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Convert vendor rates from $/CWT to $/LBS
vendor_df.loc[vendor_cwt_mask, freight_class_cols] = vendor_df.loc[vendor_cwt_mask, freight_class_cols] / 100

print("✅ Converted vendor CWT rates to $/LBS for comparability.")

✅ Converted vendor CWT rates to $/LBS for comparability.


In [145]:

# Structuring column names
combined_output = combined_output[final_cols]  # Already structured in prior step

# Append invoice summary blocks to vendor table
combined_df = pd.concat([vendor_df, combined_output], ignore_index=True)

# Preview the result
print("✅ Appended Final Table (Step 4):")
combined_df.tail()


✅ Appended Final Table (Step 4):


,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
123,SPW,SPW,CWT,Weight,1VNL,5.403926,0.214255,0.210814,0.142978,0.150177,0.140378,0.140817,0.108261,0.093772,NaN,georgia_historical_market_q3
124,SPW,SPW,CWT,Weight,1VNL,0.662675,0.177937,0.177991,0.122343,0.107951,0.096320,0.350120,0.087496,0.093772,NaN,georgia_historical_market_wavg
125,SPW,SPW,CWT,Weight,1VNL,0.836310,0.231244,0.179358,0.141366,0.141367,0.112974,0.094178,0.080727,0.058629,NaN,georgia_historical_xgs_median
126,SPW,SPW,CWT,Weight,1VNL,2.903089,0.231248,0.179359,0.141367,0.141367,0.112974,0.094178,0.089224,0.058629,NaN,georgia_historical_xgs_q3
127,SPW,SPW,CWT,Weight,1VNL,0.607491,0.231245,0.179358,0.141366,0.141367,0.112973,0.094178,0.083276,0.058629,NaN,georgia_historical_xgs_wavg


In [146]:
# Identify rows where unit is CWT and commodity is 1VNL
combined_cwt_mask = (combined_df["commodity_group"] == "1VNL") & (combined_df["unit"] == "CWT")

# List of freight class columns to scale
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Multiply vendor rates by 1.2
combined_df.loc[combined_cwt_mask, freight_class_cols] = (
    combined_df.loc[combined_cwt_mask, freight_class_cols] * 1.2
)

print("✅ Multiplied 1VNL CWT vendor rates by 1.2.")


✅ Multiplied 1VNL CWT vendor rates by 1.2.


In [147]:
combined_df['source'].unique()

array(['vendor', 'all_states_historical_market_median',
       'all_states_historical_market_q3',
       'all_states_historical_market_wavg',
       'georgia_historical_market_median', 'georgia_historical_market_q3',
       'georgia_historical_market_wavg', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_q3', 'georgia_historical_xgs_wavg'],
      dtype=object)

In [148]:
combined_df.columns

Index(['site_description', 'site', 'unit', 'unitclass', 'commodity_group',
       'L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M',
       'source'],
      dtype='object')

## Get sample size per freight class

In [149]:
# Define freight classes in correct order
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Group by site, commodity, and freight class → count unique invoice_ids
invoice_counts = invoice_all_df.groupby(
    ["site", "invoice_commodity_group", "freight_class"]
)["invoice_id"].nunique().reset_index(name="invoice_count")

# List to collect all site-level matrices
invoice_matrix_list = []

# Loop over sites
for site in invoice_counts["site"].unique():
    site_df = invoice_counts[invoice_counts["site"] == site]

    # Pivot per site
    matrix = site_df.pivot_table(
        index="invoice_commodity_group",
        columns="freight_class",
        values="invoice_count",
        fill_value=0
    )

    # Reindex to ensure all freight classes are present
    matrix = matrix.reindex(columns=freight_classes, fill_value=0)
    matrix = matrix.reset_index()

    # Add required columns
    matrix["site_description"] = "Itasca"  # Adjust dynamically if needed
    matrix["site"] = site
    matrix["unit"] = None
    matrix["unitclass"] = None
    matrix["commodity_group"] = None
    matrix["source"] = "invoice_counts"

    # Reorder to match combined_df structure
    final = matrix[[
        "site_description", "site", "unit", "unitclass",
        "commodity_group",
    ] + freight_classes + ["source"]]

    # final.rename(columns={"invoice_commodity_description": "commodity_description"}, inplace=True)
    invoice_matrix_list.append(final)

# Concatenate all site-level matrices
invoice_matrix_final = pd.concat(invoice_matrix_list, ignore_index=True)

# ✅ Now append to combined_df
combined_df = pd.concat([combined_df, invoice_matrix_final], ignore_index=True)

combined_df


,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor
1,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor
2,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor
3,Jacksonville,SPJ,SQYD,Area,1CBL,0.484997,0.472000,0.465446,0.455670,0.439341,0.439341,0.439341,0.439341,0.439341,0.439341,vendor
4,Jacksonville,SPJ,SQYD,Area,1CPT,0.824472,0.802366,0.791258,0.774706,0.746935,0.746935,0.746935,0.746935,0.746935,0.746935,vendor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Itasca,SPT,None,None,None,400.000000,42.000000,17.000000,4.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,invoice_counts
136,Itasca,SPT,None,None,None,224.000000,94.000000,70.000000,60.000000,52.000000,48.000000,14.000000,4.000000,2.000000,3.000000,invoice_counts
137,Itasca,SPW,None,None,None,74.000000,3.000000,1.000000,2.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,invoice_counts
138,Itasca,SPW,None,None,None,477.000000,39.000000,20.000000,8.000000,15.000000,3.000000,0.000000,0.000000,0.000000,0.000000,invoice_counts


## Rearrange the rates

In [150]:
summary_combined.head(2)

,site,invoice_commodity_group,all_states_historical_market_median,all_states_historical_market_wavg,all_states_historical_market_q3,georgia_historical_market_median,georgia_historical_market_wavg,georgia_historical_market_q3,georgia_historical_xgs_median,georgia_historical_xgs_wavg,georgia_historical_xgs_q3
0,SPJ,1CBL,0.72375,0.698256,1.336381,1.110548,1.031672,2.142453,0.700063,0.532134,1.231899
1,SPJ,1CPT,1.37054,1.149648,2.680562,1.365688,1.149326,2.412391,0.824475,0.762656,1.139937


In [151]:
summary_combined.columns

Index(['site', 'invoice_commodity_group',
       'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3'],
      dtype='object')

In [152]:
# Extract relevant subset
small_table = summary_combined[['site', 'invoice_commodity_group',
       'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3']].copy()

# Melt into long format
small_table_long = small_table.melt(
    id_vars=["site", "invoice_commodity_group"],
    value_vars=[
        'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3'
    ],
    var_name="source",
    value_name="ppt_rates"
)

# ✅ Rename the column
small_table_long = small_table_long.rename(columns={"invoice_commodity_group": "commodity_group"})

# Preview
print(small_table_long.head())


  site commodity_group                               source  ppt_rates
0  SPJ            1CBL  all_states_historical_market_median   0.723750
1  SPJ            1CPT  all_states_historical_market_median   1.370540
2  SPJ            1VNL  all_states_historical_market_median   0.187061
3  SPN            1CBL  all_states_historical_market_median   0.854848
4  SPN            1CPT  all_states_historical_market_median   0.789551


In [153]:
merged_df = pd.merge(
    combined_df,
    small_table_long,
    on=["site", "commodity_group", "source"],
    how="left"
)
merged_df

,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source,ppt_rates
0,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
1,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
2,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
3,Jacksonville,SPJ,SQYD,Area,1CBL,0.484997,0.472000,0.465446,0.455670,0.439341,0.439341,0.439341,0.439341,0.439341,0.439341,vendor,NaN
4,Jacksonville,SPJ,SQYD,Area,1CPT,0.824472,0.802366,0.791258,0.774706,0.746935,0.746935,0.746935,0.746935,0.746935,0.746935,vendor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Itasca,SPT,None,None,None,400.000000,42.000000,17.000000,4.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,invoice_counts,NaN
136,Itasca,SPT,None,None,None,224.000000,94.000000,70.000000,60.000000,52.000000,48.000000,14.000000,4.000000,2.000000,3.000000,invoice_counts,NaN
137,Itasca,SPW,None,None,None,74.000000,3.000000,1.000000,2.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,invoice_counts,NaN
138,Itasca,SPW,None,None,None,477.000000,39.000000,20.000000,8.000000,15.000000,3.000000,0.000000,0.000000,0.000000,0.000000,invoice_counts,NaN


In [154]:
import pandas as pd

# Define source mappings
median_sources = [
    "georgia_historical_market_median",
    "all_states_historical_market_median",
    "georgia_historical_xgs_median"
]
wavg_sources = [
    "georgia_historical_market_wavg",
    "all_states_historical_market_wavg",
    "georgia_historical_xgs_wavg"
]
q3_sources = [
    "georgia_historical_market_q3",
    "all_states_historical_market_q3",
    "georgia_historical_xgs_q3"
]

freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

output_rows = []

for group_key, group_df in merged_df.groupby(['site_description', 'site', 'unit', 'unitclass', 'commodity_group']):
    output_rows.append(group_df)

    has_any_valid_source = group_df["source"].isin(
        median_sources + wavg_sources + q3_sources
    ).any()
    if not has_any_valid_source:
        continue

    # Subsets
    median_df = group_df[group_df["source"].isin(median_sources)]
    wavg_df = group_df[group_df["source"].isin(wavg_sources)]
    q3_df = group_df[group_df["source"].isin(q3_sources)]

    base = group_df.iloc[0].copy()
    base[:] = None

    def make_row(label, df_subset):
        r = base.copy()
        r["source"] = label
        if not df_subset.empty:
            r["ppt_rates"] = df_subset["ppt_rates"].max() * 1.06
            for col in freight_class_cols:
                r[col] = df_subset[col].max() * 1.06
        for col, val in zip(['site_description', 'site', 'unit', 'unitclass', 'commodity_group'], group_key):
            r[col] = val
        return r

    # Step 1: Base recommended rows
    row_median = make_row("recommended_median", median_df)
    row_wavg = make_row("recommended_wavg", wavg_df)
    row_q3 = make_row("recommended_q3", q3_df)

    output_rows.append(pd.DataFrame([row_median, row_wavg, row_q3]))

    # Step 2: Create ranked versions (min/middle/max)
    recommended_values = [
        row_median["ppt_rates"],
        row_wavg["ppt_rates"],
        row_q3["ppt_rates"]
    ]

    if all(pd.notna(recommended_values)):
        sorted_values = sorted(recommended_values)
        ranked_labels = ["recommended_min", "recommended_middle", "recommended_max"]

        def make_rank_row(label, value):
            r = base.copy()
            r["source"] = label
            r["ppt_rates"] = value
            for col, val in zip(['site_description', 'site', 'unit', 'unitclass', 'commodity_group'], group_key):
                r[col] = val
            return r

        output_rows.append(pd.DataFrame([
            make_rank_row(ranked_labels[0], sorted_values[0]),
            make_rank_row(ranked_labels[1], sorted_values[1]),
            make_rank_row(ranked_labels[2], sorted_values[2]),
        ]))

# Final DataFrame
final_df = pd.concat(output_rows, ignore_index=True)

print("✅ Final DataFrame includes full recommendations and ranked (min/middle/max) recommendations.")


✅ Final DataFrame includes full recommendations and ranked (min/middle/max) recommendations.


In [155]:
# Final Sort: Enforce output row order for readability

# Define source display order
source_order = {
    "vendor": 1,

    # Georgia - historical market
    "georgia_historical_market_median": 2,
    "georgia_historical_market_wavg": 3,
    "georgia_historical_market_q3": 4,

    # All states - historical market
    "all_states_historical_market_median": 5,
    "all_states_historical_market_wavg": 6,
    "all_states_historical_market_q3": 7,

    # Georgia - xgs
    "georgia_historical_xgs_median": 8,
    "georgia_historical_xgs_wavg": 9,
    "georgia_historical_xgs_q3": 10,

    # All states - xgs
    "all_states_historical_xgs_median": 11,
    "all_states_historical_xgs_wavg": 12,
    "all_states_historical_xgs_q3": 13,

       # All states - xgs
    "recommended_median": 14,
    "recommended_wavg": 15,
    "recommended_q3": 16,
         # All states - xgs
    "recommended_min": 17,
    "recommended_middle": 18,
    "recommended_max": 19,

    "invoice_counts": 99  # Always last
}

# Add sorting key column
final_df["source_sort"] = final_df["source"].map(source_order)

# Sort rows to follow commodity hierarchy and defined source order
final_df = final_df.sort_values(
    by=["commodity_group", "site", "unit", "source_sort"]
).drop(columns="source_sort")

# Reset index for cleanliness
final_df.reset_index(drop=True, inplace=True)

print("✅ Rows sorted for visual clarity.")
display(final_df.head(20))  # Display first 20 rows for quick check


✅ Rows sorted for visual clarity.


,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source,ppt_rates
0,Jacksonville,SPJ,SQYD,Area,1CBL,0.484997,0.472000,0.465446,0.455670,0.439341,0.439341,0.439341,0.439341,0.439341,0.439341,vendor,NaN
1,SPJ,SPJ,SQYD,Area,1CBL,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median,1.110548
2,SPJ,SPJ,SQYD,Area,1CBL,1.243128,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_wavg,1.031672
3,SPJ,SPJ,SQYD,Area,1CBL,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3,2.142453
4,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median,0.723750
5,SPJ,SPJ,SQYD,Area,1CBL,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg,0.698256
6,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3,1.336381
7,SPJ,SPJ,SQYD,Area,1CBL,0.811437,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_median,0.700063
8,SPJ,SPJ,SQYD,Area,1CBL,0.700245,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_wavg,0.532134
9,SPJ,SPJ,SQYD,Area,1CBL,1.385038,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_q3,1.231899


## Final rates table

In [156]:
# 🔄 Save combined_df with each site as a separate Excel sheet

import pandas as pd

# Set export path
output_path = "output/20062025_freight_rates_by_site_nigel_v1400.xlsx"  # Change path if needed

# Get unique sites
sites = final_df["site"].dropna().unique()

# Export to Excel with one sheet per site
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for site in sites:
        sheet_name = str(site)[:31]  # Excel sheet names must be ≤ 31 characters
        site_df = final_df[final_df["site"] == site]
        site_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Exported to {output_path} with one sheet per site.")


✅ Exported to output/20062025_freight_rates_by_site_nigel_v1400.xlsx with one sheet per site.


In [157]:
# # Use vendor_df to create a mapping from site to site_description
# site_desc_map = vendor_df.set_index("site")["site_description"].to_dict()

# # Update site_description in all relevant DataFrames by matching on 'site'
# for df_name in ["hist_wavg_block", "xgs_avg_block", "xgs_wavg_block", "variance_data", "combined_df", "invoice_matrix_final"]:
#     df = globals()[df_name]
#     df["site_description"] = df["site"].map(site_desc_map).fillna(df["site_description"])
# display(combined_df.tail(20))

In [158]:
# Step X: Append Variance Rows Between hist_invoice and xgs_invoice

# # Columns used to match rows
# index_cols = [
#     "site_description", "site", "unit", "unitclass",
#     "commodity_group", "commodity_description"
# ]

# # Freight class columns to compute variance on
# freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# # Separate historical and xgs rows
# hist_df = combined_df[combined_df["source"] == "hist_wavg"]
# xgs_df = combined_df[combined_df["source"] == "xgs_wavg"]

# # Merge them on the index columns
# variance_df = pd.merge(hist_df, xgs_df, on=index_cols, suffixes=("_hist", "_xgs"))

# # Compute variance
# variance_data = variance_df[index_cols].copy()
# for col in freight_class_cols:
#     variance_data[col] = variance_df[f"{col}_hist"] - variance_df[f"{col}_xgs"]

# # Add source column
# variance_data["source"] = "variance"

# # Append to combined table
# combined_df = pd.concat([combined_df, variance_data], ignore_index=True)

# # Optional: sort for clarity
# combined_df.sort_values(by=index_cols + ["source"], inplace=True)

# # Preview result
# print("✅ Variance rows added.")
# combined_df.tail()
